<a href="https://colab.research.google.com/github/AbdulUMSL/EENG-1108/blob/main/OsmanMidterm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import string

WORDLIST_FILENAME = "words.txt"


def load_words():
    """
    Returns a list of valid words. Words are strings of lowercase letters.
    """
    print("Loading word list from file...")
    with open(WORDLIST_FILENAME, 'r') as inFile:
        line = inFile.readline()
        wordlist = line.split()
    print(len(wordlist), "words loaded.")
    return wordlist


def choose_word(wordlist):
    """
    wordlist (list): list of words (strings)
    Returns a word from wordlist at random
    """
    return random.choice(wordlist)


# Load word list
wordlist = load_words()


# -----------------------------------
# Part 1 Helper Functions
# -----------------------------------

def is_word_guessed(secret_word, letters_guessed):
    """
    secret_word: string, the word the user is guessing
    letters_guessed: list of letters guessed so far
    returns: boolean, True if all letters of secret_word are in letters_guessed
    """
    for letter in secret_word:
        if letter not in letters_guessed:
            return False
    return True


def get_guessed_word(secret_word, letters_guessed):
    """
    secret_word: string, the word the user is guessing
    letters_guessed: list of letters guessed so far
    returns: string comprised of letters, underscores (_), and spaces that
             represents which letters in secret_word have been guessed so far
    """
    result = ""
    for letter in secret_word:
        if letter in letters_guessed:
            result += letter
        else:
            result += "_ "
    return result


def get_available_letters(letters_guessed):
    """
    letters_guessed: list of letters guessed so far
    returns: string comprised of letters that represents which letters have
             not yet been guessed
    """
    result = ""
    for letter in string.ascii_lowercase:
        if letter not in letters_guessed:
            result += letter
    return result


# -----------------------------------
# Part 2 Main Hangman Game
# -----------------------------------

def hangman(secret_word):
    """
    secret_word: string, the secret word to guess.
    Starts up an interactive game of Hangman.
    """
    guesses_remaining = 6
    warnings_remaining = 3
    letters_guessed = []

    print("Welcome to the game Hangman!")
    print("I am thinking of a word that is", len(secret_word), "letters long.")
    print("You have", warnings_remaining, "warnings left.")
    print("-------------")

    while guesses_remaining > 0 and not is_word_guessed(secret_word, letters_guessed):

        print("You have", guesses_remaining, "guesses left.")
        print("Available letters:", get_available_letters(letters_guessed))

        guess = input("Please guess a letter: ").lower()

        # Invalid input — not a single alpha character
        if not guess.isalpha() or len(guess) != 1:
            if warnings_remaining > 0:
                warnings_remaining -= 1
                print("Oops! That is not a valid letter. You have",
                      warnings_remaining, "warnings left:",
                      get_guessed_word(secret_word, letters_guessed))
            else:
                guesses_remaining -= 1
                print("Oops! That is not a valid letter. You have no warnings left "
                      "so you lose one guess:",
                      get_guessed_word(secret_word, letters_guessed))

        # Already guessed
        elif guess in letters_guessed:
            if warnings_remaining > 0:
                warnings_remaining -= 1
                # FIX: spec says "You now have X warnings" not "You have X warnings left"
                print("Oops! You've already guessed that letter. You now have",
                      warnings_remaining, "warnings:",
                      get_guessed_word(secret_word, letters_guessed))
            else:
                guesses_remaining -= 1
                print("Oops! You've already guessed that letter. You have no warnings left "
                      "so you lose one guess:",
                      get_guessed_word(secret_word, letters_guessed))

        # New valid guess
        else:
            letters_guessed.append(guess)
            if guess in secret_word:
                print("Good guess:", get_guessed_word(secret_word, letters_guessed))
            else:
                print("Oops! That letter is not in my word:",
                      get_guessed_word(secret_word, letters_guessed))
                # Vowels cost 2 guesses, consonants cost 1
                if guess in "aeiou":
                    guesses_remaining -= 2
                else:
                    guesses_remaining -= 1

        print("-------------")

    # Game End
    if is_word_guessed(secret_word, letters_guessed):
        unique_letters = len(set(secret_word))
        score = guesses_remaining * unique_letters
        print("Congratulations, you won!")
        print("Your total score for this game is:", score)
    else:
        print("Sorry, you ran out of guesses. The word was", secret_word)


# -----------------------------------
# Part 3 Hints Mode
# -----------------------------------

def match_with_gaps(my_word, other_word):
    """
    my_word: string with _ characters, current guess state (e.g. 't_ _ t')
    other_word: string, regular English word
    returns: boolean, True if my_word matches other_word pattern
    """
    # FIX: Use replace to remove spaces from '_ ' representations
    my_word = my_word.replace(" ", "")

    if len(my_word) != len(other_word):
        return False

    # FIX: Build the set of already-revealed letters for proper blank checking
    # A blank position must NOT correspond to a letter already revealed in my_word
    revealed_letters = set(c for c in my_word if c != "_")

    for i in range(len(my_word)):
        if my_word[i] == "_":
            # The blank cannot be filled by a letter we've already revealed
            # (because if it were that letter, it would have been shown, not hidden)
            if other_word[i] in revealed_letters:
                return False
        else:
            # Revealed positions must match exactly
            if my_word[i] != other_word[i]:
                return False

    return True


def show_possible_matches(my_word):
    """
    my_word: string with _ characters, current guess state
    Prints all words in wordlist that match my_word pattern.
    """
    matches = []

    for word in wordlist:
        if match_with_gaps(my_word, word):
            matches.append(word)

    if len(matches) == 0:
        print("No matches found")
    else:
        print("Possible word matches are:")
        print(" ".join(matches))  # FIX: cleaner than loop with end=" "


def hangman_with_hints(secret_word):
    """
    secret_word: string, the secret word to guess.
    Hangman game with the addition that if the user types '*', they receive
    a list of all possible matching words.
    """
    guesses_remaining = 6
    warnings_remaining = 3
    letters_guessed = []

    print("Welcome to the game Hangman!")
    print("I am thinking of a word that is", len(secret_word), "letters long.")
    print("You have", warnings_remaining, "warnings left.")
    print("-------------")

    while guesses_remaining > 0 and not is_word_guessed(secret_word, letters_guessed):

        print("You have", guesses_remaining, "guesses left.")
        print("Available letters:", get_available_letters(letters_guessed))

        guess = input("Please guess a letter: ").lower()

        # HINT: asterisk triggers possible matches — no penalty
        if guess == "*":
            show_possible_matches(get_guessed_word(secret_word, letters_guessed))
            print("-------------")
            continue

        # Invalid input
        if not guess.isalpha() or len(guess) != 1:
            if warnings_remaining > 0:
                warnings_remaining -= 1
                print("Oops! That is not a valid letter. You have",
                      warnings_remaining, "warnings left:",
                      get_guessed_word(secret_word, letters_guessed))
            else:
                guesses_remaining -= 1
                print("Oops! That is not a valid letter. You have no warnings left "
                      "so you lose one guess:",
                      get_guessed_word(secret_word, letters_guessed))

        # Already guessed
        elif guess in letters_guessed:
            if warnings_remaining > 0:
                warnings_remaining -= 1
                # FIX: wording matches spec
                print("Oops! You've already guessed that letter. You now have",
                      warnings_remaining, "warnings:",
                      get_guessed_word(secret_word, letters_guessed))
            else:
                guesses_remaining -= 1
                print("Oops! You've already guessed that letter. You have no warnings left "
                      "so you lose one guess:",
                      get_guessed_word(secret_word, letters_guessed))

        # New valid guess
        else:
            letters_guessed.append(guess)
            if guess in secret_word:
                print("Good guess:", get_guessed_word(secret_word, letters_guessed))
            else:
                print("Oops! That letter is not in my word:",
                      get_guessed_word(secret_word, letters_guessed))
                if guess in "aeiou":
                    guesses_remaining -= 2
                else:
                    guesses_remaining -= 1

        print("-------------")

    # Game End
    if is_word_guessed(secret_word, letters_guessed):
        unique_letters = len(set(secret_word))
        score = guesses_remaining * unique_letters
        print("You won!")
        print("Your total score for this game is:", score)
    else:
        print("Sorry, you ran out of guesses. The word was", secret_word)


if __name__ == "__main__":
    secret_word = choose_word(wordlist)
    hangman_with_hints(secret_word)